# Agent 常用开发框架详解

## 框架全景

| 框架 | 定位 | 核心能力 | 学习曲线 |
|------|------|---------|----------|
| LangChain | 通用Agent框架 | 工具调用/链式编排/RAG | 中等 |
| LlamaIndex | RAG专家 | 数据索引/检索增强 | 低 |
| AutoGen | 多Agent协作 | 对话式多Agent | 中等 |
| CrewAI | 角色扮演 | 团队协作/任务分配 | 低 |
| Semantic Kernel | 企业级 | 插件化/多语言 | 中等 |
| Dify | 低代码平台 | 可视化编排 | 很低 |

## 本课内容
1. LangChain 核心概念与实战
2. LlamaIndex RAG实战
3. AutoGen 多Agent协作
4. CrewAI 角色扮演
5. 框架选型指南
6. 从零构建轻量Agent框架

In [ ]:
import json
import re
import numpy as np
from typing import List, Dict, Optional, Callable, Any
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
print("环境准备完成")

## 1. LangChain 核心概念

LangChain 是最流行的Agent框架, 核心抽象:

```
LangChain 核心组件:
├── Model I/O       # LLM接口
│   ├── Prompts     # 提示词模板
│   ├── LLMs        # 语言模型
│   └── Output Parsers # 输出解析
├── Retrieval       # 检索
│   ├── Document Loaders # 文档加载
│   ├── Text Splitters  # 文本分割
│   ├── Embeddings      # 向量化
│   └── Vector Stores   # 向量存储
├── Chains          # 链式调用
├── Agents          # 智能体
│   ├── Tools       # 工具
│   ├── Toolkits    # 工具包
│   └── Agent Types # Agent类型
└── Memory          # 记忆
```

In [ ]:
# LangChain 核心概念模拟 (无需安装即可理解)

# 1. Prompt Template
class PromptTemplate:
    def __init__(self, template: str, input_variables: List[str]):
        self.template = template
        self.input_variables = input_variables
    
    def format(self, **kwargs) -> str:
        result = self.template
        for key, value in kwargs.items():
            result = result.replace("{" + key + "}", str(value))
        return result

# RFFI Agent 系统提示词
rffi_prompt = PromptTemplate(
    template="""你是一个射频指纹识别(RFFI)专家。
你可以使用以下工具:
{tools}

请按以下格式回答:
Question: 输入问题
Thought: 思考过程
Action: 工具名称(参数)
Observation: 工具返回结果
... (可以多轮Thought/Action/Observation)
Thought: 我已经知道最终答案
Final Answer: 最终答案

开始!
Question: {question}
Thought: {agent_scratchpad}""",
    input_variables=["tools", "question", "agent_scratchpad"]
)

formatted = rffi_prompt.format(
    tools="1. rf_scan: 扫描射频信号\n2. rf_fingerprint: 提取射频指纹",
    question="识别2.4GHz频段的设备",
    agent_scratchpad=""
)
print(formatted[:500])

In [ ]:
# 2. Chain (链式调用)
class SimpleChain:
    def __init__(self, steps: List[Callable]):
        self.steps = steps
    
    def run(self, input_data: Any) -> Any:
        result = input_data
        for step in self.steps:
            result = step(result)
        return result

# RFFI处理链
def load_iq(raw: str) -> Dict:
    return {"raw": raw, "iq": np.random.randn(2, 128).astype(np.float32)}

def preprocess(data: Dict) -> Dict:
    iq = data["iq"]
    iq = iq / (np.std(iq) + 1e-8)
    data["iq_normalized"] = iq
    return data

def predict(data: Dict) -> Dict:
    np.random.seed(42)
    data["device_id"] = np.random.randint(0, 5)
    data["confidence"] = np.random.uniform(0.8, 0.99)
    return data

def format_output(data: Dict) -> str:
    return f"Device {data['device_id']}, Confidence {data['confidence']:.4f}"

rffi_chain = SimpleChain([load_iq, preprocess, predict, format_output])
result = rffi_chain.run("raw_iq_data_hex_string")
print(f"Chain 结果: {result}")
print()
print("LangChain Chain 类型:")
print("  LLMChain:      LLM + Prompt (最基础)")
print("  SequentialChain: 顺序执行多个Chain")
print("  RouterChain:   根据输入路由到不同Chain")
print("  TransformChain: 自定义转换函数")

In [ ]:
# 3. LangChain Agent 类型
print("=== LangChain Agent 类型 ===")
print()
print("1. ReAct Agent")
print("   最通用, Thought-Action-Observation循环")
print("   适合: 需要多步推理的任务")
print()
print("2. OpenAI Functions Agent")
print("   利用OpenAI的Function Calling")
print("   适合: 使用GPT模型的场景, 更可靠")
print()
print("3. Structured Chat Agent")
print("   支持多输入工具, 结构化对话")
print("   适合: 工具参数复杂的场景")
print()
print("4. Plan-and-Execute Agent")
print("   先规划再执行, 适合长任务")
print("   适合: 复杂多步骤任务")
print()
print("5. Self-Ask Agent")
print("   自我提问, 追踪式推理")
print("   适合: 需要查找多步信息的问答")
print()
print("选择建议:")
print("  GPT模型 + 简单工具 -> OpenAI Functions Agent")
print("  开源模型 + 通用场景 -> ReAct Agent")
print("  复杂多步任务       -> Plan-and-Execute Agent")

In [ ]:
# 4. LangChain 实战代码模板
print("=== LangChain RFFI Agent 代码模板 ===")
print()
template = '''
from langchain_openai import ChatOpenAI
from langchain.agents import create_openai_tools_agent, AgentExecutor
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 1. 定义工具
@tool
def rf_scan(frequency_mhz: float) -> str:
    """扫描指定频率的射频信号"""
    # 实际实现: 调用SDR API
    return json.dumps({"devices_found": 3, "frequency": frequency_mhz})

@tool
def rf_fingerprint(device_id: int) -> str:
    """提取设备射频指纹并识别"""
    # 实际实现: 调用RFFI模型API
    return json.dumps({"device_id": device_id, "confidence": 0.95})

@tool
def check_auth_policy(device_id: int, confidence: float) -> str:
    """检查设备是否符合认证策略"""
    if confidence >= 0.7:
        return f"Device {device_id} AUTHORIZED"
    return f"Device {device_id} REJECTED (low confidence)"

tools = [rf_scan, rf_fingerprint, check_auth_policy]

# 2. 创建Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是射频安全专家, 负责设备发现和认证。"),
    ("user", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

# 3. 创建Agent
llm = ChatOpenAI(model="gpt-4", temperature=0)
agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 4. 运行
result = agent_executor.invoke({
    "input": "扫描2.4GHz频段, 识别发现的设备, 并检查认证策略"
})
print(result["output"])
'''
print(template)

---
## 2. LlamaIndex RAG实战

LlamaIndex 专注数据检索, RAG能力最强。

```
LlamaIndex 核心流程:

数据源 -> 文档 -> 节点 -> 索引 -> 查询引擎 -> 回答
  |        |       |       |          |
PDF/DB   Document  Node  VectorIndex  QueryEngine
Web/API   对象    分块    向量索引     检索+生成
```

In [ ]:
# LlamaIndex 核心概念模拟

@dataclass
class Document:
    doc_id: str
    text: str
    metadata: Dict = field(default_factory=dict)

@dataclass
class Node:
    node_id: str
    text: str
    embedding: Optional[np.ndarray] = None
    metadata: Dict = field(default_factory=dict)

class SimpleIndex:
    def __init__(self):
        self.nodes: List[Node] = []
    
    def add_nodes(self, nodes: List[Node]):
        self.nodes.extend(nodes)
    
    def _embed(self, text: str) -> np.ndarray:
        np.random.seed(hash(text) % 2**31)
        return np.random.randn(64).astype(np.float32)
    
    def query(self, query_text: str, top_k: int = 3) -> List[Node]:
        q_embed = self._embed(query_text)
        scored = []
        for node in self.nodes:
            if node.embedding is None:
                node.embedding = self._embed(node.text)
            score = np.dot(q_embed, node.embedding) / (
                np.linalg.norm(q_embed) * np.linalg.norm(node.embedding) + 1e-8)
            scored.append((score, node))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [node for _, node in scored[:top_k]]

class TextSplitter:
    def __init__(self, chunk_size: int = 200, overlap: int = 50):
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def split(self, doc: Document) -> List[Node]:
        text = doc.text
        nodes = []
        start = 0
        idx = 0
        while start < len(text):
            end = min(start + self.chunk_size, len(text))
            nodes.append(Node(
                node_id=f"{doc.doc_id}_{idx}",
                text=text[start:end],
                metadata={**doc.metadata, "chunk": idx}
            ))
            start += self.chunk_size - self.overlap
            idx += 1
        return nodes

# 构建RFFI知识索引
docs = [
    Document("doc1", "射频指纹识别(RFFI)通过提取无线设备的硬件特征来识别设备身份。主要指纹来源包括载波频偏(CFO)、IQ不平衡、功率放大器非线性。CFO由振荡器偏差引起, 不同设备的CFO不同。", {"topic": "rffi_basics"}),
    Document("doc2", "1D CNN是RFFI最常用的模型架构。输入为IQ信号(2,N), 通过多层1D卷积提取时域特征。典型结构: Conv1d-BN-ReLU-MaxPool-FC。ResNet1D和SE-Net可以进一步提升性能。", {"topic": "model"}),
    Document("doc3", "Open-set识别是RFFI的关键挑战, 需要拒绝未知设备。度量学习(ArcFace, Center Loss)比分类损失更适合Open-set场景。阈值法是最简单的基线方法。", {"topic": "open_set"}),
    Document("doc4", "域适应解决跨环境泛化问题。迁移学习在目标域少量样本上微调。MMD和DANN是无监督域适应方法。多SNR联合训练可以提高鲁棒性。", {"topic": "domain_adaptation"}),
]

splitter = TextSplitter(chunk_size=150, overlap=30)
index = SimpleIndex()

for doc in docs:
    nodes = splitter.split(doc)
    index.add_nodes(nodes)

print(f"文档数: {len(docs)}, 节点数: {len(index.nodes)}")

results = index.query("如何解决Open-set识别问题", top_k=3)
print(f"\n查询: 如何解决Open-set识别问题")
for r in results:
    print(f"  [{r.node_id}] {r.text[:60]}...")

In [ ]:
# LlamaIndex 代码模板
print("=== LlamaIndex RFFI RAG 代码模板 ===")
print()
template = '''
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter

# 1. 配置
Settings.llm = OpenAI(model="gpt-4", temperature=0)
Settings.embed_model = OpenAIEmbedding()

# 2. 加载文档
documents = SimpleDirectoryReader("./rffi_knowledge/").load_data()

# 3. 分块
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

# 4. 构建索引
index = VectorStoreIndex(nodes)

# 5. 查询
query_engine = index.as_query_engine(similarity_top_k=3)
response = query_engine.query("1D CNN如何用于射频指纹识别?")
print(response)

# 6. 作为Agent工具
from llama_index.core.tools import QueryEngineTool
rffi_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="rffi_knowledge",
    description="射频指纹识别知识库, 可查询RFFI相关技术"
)
'''
print(template)

---
## 3. AutoGen 多Agent协作

AutoGen (微软) 专注多Agent对话式协作。

```
AutoGen 核心概念:

AssistantAgent:  LLM驱动的助手
UserProxyAgent: 代表用户的代理
GroupChat:      多Agent群聊

协作模式:
  1v1: 用户 <-> 助手
  群聊: 多个Agent在群组中讨论
  层级: 管理者分配任务给专家
```

In [ ]:
# AutoGen 模拟
class AutoGenAgent:
    def __init__(self, name: str, system_message: str, role: str = "assistant"):
        self.name = name
        self.system_message = system_message
        self.role = role
        self.messages: List[Dict] = [{"role": "system", "content": system_message}]
    
    def receive(self, message: str, sender: str):
        self.messages.append({"role": "user", "content": f"[{sender}]: {message}"})
    
    def respond(self) -> str:
        last_msg = self.messages[-1]["content"]
        return self._generate_response(last_msg)
    
    def _generate_response(self, input_msg: str) -> str:
        np.random.seed(hash(input_msg + self.name) % 2**31)
        if self.name == "Scanner":
            n = np.random.randint(2, 6)
            return f"扫描完成, 发现{n}个设备: Device-0到Device-{n-1}, 频率2.4GHz"
        elif self.name == "Authenticator":
            conf = np.random.uniform(0.7, 0.99)
            dev = np.random.randint(0, 5)
            return f"设备Device-{dev}认证结果: 置信度{conf:.4f}, {'通过' if conf > 0.7 else '拒绝'}"
        elif self.name == "SecurityAnalyst":
            threats = np.random.choice([0, 1])
            return f"安全分析: 发现{threats}个潜在威胁, 风险等级{'低' if threats == 0 else '中'}"
        elif self.name == "Manager":
            return "请Scanner扫描设备, Authenticator进行认证, SecurityAnalyst检查安全"
        return "收到"

class GroupChat:
    def __init__(self, agents: List[AutoGenAgent], max_rounds: int = 6):
        self.agents = agents
        self.max_rounds = max_rounds
        self.chat_history: List[Dict] = []
    
    def run(self, task: str) -> List[Dict]:
        self.chat_history.append({"speaker": "User", "message": task})
        
        for round_idx in range(self.max_rounds):
            agent = self.agents[round_idx % len(self.agents)]
            
            context = "\n".join([f"[{h['speaker']}]: {h['message']}" for h in self.chat_history[-3:]])
            agent.receive(context, "GroupChat")
            response = agent.respond()
            
            self.chat_history.append({"speaker": agent.name, "message": response})
        
        return self.chat_history

# 创建Agent团队
scanner = AutoGenAgent("Scanner", "你是射频扫描专家, 负责发现和监测无线设备")
authenticator = AutoGenAgent("Authenticator", "你是设备认证专家, 负责射频指纹识别和认证")
security = AutoGenAgent("SecurityAnalyst", "你是安全分析专家, 负责威胁检测")

group = GroupChat([scanner, authenticator, security], max_rounds=6)
history = group.run("对当前2.4GHz环境进行安全审计")

print("=== AutoGen 群聊记录 ===")
for h in history:
    print(f"[{h['speaker']}]: {h['message'][:80]}")

In [ ]:
# AutoGen 代码模板
print("=== AutoGen 代码模板 ===")
print()
template = '''
import autogen

# 1. 配置LLM
config_list = [{"model": "gpt-4", "api_key": "your-key"}]
llm_config = {"config_list": config_list}

# 2. 创建Agent
scanner = autogen.AssistantAgent(
    name="Scanner",
    system_message="你是射频扫描专家",
    llm_config=llm_config,
)

authenticator = autogen.AssistantAgent(
    name="Authenticator",
    system_message="你是设备认证专家",
    llm_config=llm_config,
)

user_proxy = autogen.UserProxyAgent(
    name="User",
    human_input_mode="NEVER",  # 自动模式
    code_execution_config={"use_docker": False},
)

# 3. 群聊
groupchat = autogen.GroupChat(
    agents=[user_proxy, scanner, authenticator],
    messages=[], max_round=10
)
manager = autogen.GroupChatManager(groupchat=groupchat, llm_config=llm_config)

# 4. 启动
user_proxy.initiate_chat(
    manager, message="扫描2.4GHz频段并认证发现的设备"
)
'''
print(template)

---
## 4. CrewAI 角色扮演

CrewAI 用角色扮演的方式组织Agent, 更直观:

```
CrewAI 核心概念:

Agent:  角色 + 目标 + 背景 + 工具
Task:   描述 + 负责Agent + 期望输出
Crew:   Agent团队 + 任务列表 + 流程类型

流程类型:
  Sequential: 按顺序执行任务
  Hierarchical: 管理者分配任务
```

In [ ]:
# CrewAI 模拟
@dataclass
class CrewAgent:
    role: str
    goal: str
    backstory: str
    tools: List[str] = field(default_factory=list)
    
    def execute(self, task_desc: str) -> str:
        np.random.seed(hash(task_desc + self.role) % 2**31)
        if "scan" in self.role.lower() or "scanner" in self.role.lower():
            n = np.random.randint(2, 6)
            return f"扫描报告: 发现{n}个无线设备, 2个WiFi, 1个BLE, {n-3}个LoRa"
        elif "auth" in self.role.lower():
            return f"认证报告: 3个设备通过认证, 1个设备置信度不足被拒绝"
        elif "security" in self.role.lower() or "analyst" in self.role.lower():
            return f"安全报告: 1个未授权设备, 建议加强监控"
        return f"{self.role}完成任务"

@dataclass
class CrewTask:
    description: str
    agent: CrewAgent
    expected_output: str

class Crew:
    def __init__(self, agents: List[CrewAgent], tasks: List[CrewTask], process: str = "sequential"):
        self.agents = agents
        self.tasks = tasks
        self.process = process
    
    def kickoff(self) -> Dict:
        results = []
        for task in self.tasks:
            result = task.agent.execute(task.description)
            results.append({"task": task.description, "agent": task.agent.role, "result": result})
        return {"process": self.process, "results": results}

# 定义RFFI安全审计团队
scanner = CrewAgent(
    role="RF Scanner",
    goal="发现和监测无线设备",
    backstory="你是一名经验丰富的射频工程师, 擅长频谱分析和设备发现",
    tools=["spectrum_analyzer", "sdr_receiver"]
)

authenticator = CrewAgent(
    role="Device Authenticator",
    goal="通过射频指纹识别和认证设备身份",
    backstory="你是射频指纹识别专家, 精通1D-CNN和度量学习",
    tools=["rffi_model", "fingerprint_db"]
)

analyst = CrewAgent(
    role="Security Analyst",
    goal="检测安全威胁并生成报告",
    backstory="你是IoT安全专家, 专注于物理层安全分析",
    tools=["threat_db", "anomaly_detector"]
)

tasks = [
    CrewTask("扫描2.4GHz频段, 发现所有无线设备", scanner, "设备列表和信号特征"),
    CrewTask("对发现的设备进行射频指纹认证", authenticator, "认证结果和置信度"),
    CrewTask("分析安全风险并生成综合报告", analyst, "安全评估报告"),
]

crew = Crew(agents=[scanner, authenticator, analyst], tasks=tasks, process="sequential")
result = crew.kickoff()

print("=== CrewAI 执行结果 ===")
print(f"流程: {result['process']}")
for r in result['results']:
    print(f"\n[{r['agent']}]")
    print(f"  任务: {r['task']}")
    print(f"  结果: {r['result']}")

In [ ]:
# CrewAI 代码模板
print("=== CrewAI 代码模板 ===")
print()
template = '''
from crewai import Agent, Task, Crew, Process

# 1. 定义Agent
scanner = Agent(
    role="RF Scanner",
    goal="发现和监测无线设备",
    backstory="你是射频工程师, 擅长频谱分析",
    verbose=True,
    allow_delegation=False,
)

authenticator = Agent(
    role="Device Authenticator",
    goal="通过射频指纹认证设备",
    backstory="你是RFFI专家, 精通深度学习",
    verbose=True,
    allow_delegation=False,
)

analyst = Agent(
    role="Security Analyst",
    goal="检测威胁并生成报告",
    backstory="你是IoT安全专家",
    verbose=True,
    allow_delegation=True,  # 可以委派任务
)

# 2. 定义Task
scan_task = Task(
    description="扫描2.4GHz频段, 发现所有无线设备",
    agent=scanner,
    expected_output="设备列表和信号特征"
)

auth_task = Task(
    description="对发现的设备进行射频指纹认证",
    agent=authenticator,
    expected_output="认证结果和置信度"
)

report_task = Task(
    description="综合分析并生成安全报告",
    agent=analyst,
    expected_output="安全评估报告"
)

# 3. 创建Crew
crew = Crew(
    agents=[scanner, authenticator, analyst],
    tasks=[scan_task, auth_task, report_task],
    process=Process.sequential,  # 或 Process.hierarchical
)

# 4. 运行
result = crew.kickoff()
print(result)
'''
print(template)

---
## 5. 框架选型指南

In [ ]:
# 框架详细对比
print("=== 框架详细对比 ===")
print()
dims = ["定位", "核心优势", "不足", "RAG能力", "Multi-Agent", "生态", "上手难度", "RFFI适配"]
langchain = ["通用Agent框架", "生态最丰富", "复杂/不稳定", "强", "支持", "最大", "中等", "★★★★"]
llamaindex = ["RAG专家", "检索能力最强", "Agent能力弱", "最强", "有限", "中等", "低", "★★★★★"]
autogen   = ["多Agent协作", "对话式协作", "单Agent弱", "弱", "最强", "中等", "中等", "★★★"]
crewai    = ["角色扮演", "直观易用", "定制性弱", "弱", "强", "较小", "低", "★★★"]
sk        = ["企业级", "多语言/稳定", "Python支持弱", "中等", "支持", "大", "中等", "★★"]
dify      = ["低代码平台", "可视化/快速", "定制性弱", "中等", "支持", "中等", "很低", "★★"]

frameworks = ["LangChain", "LlamaIndex", "AutoGen", "CrewAI", "Semantic Kernel", "Dify"]
data = [langchain, llamaindex, autogen, crewai, sk, dify]

for d_name, d_data in zip(dims, zip(*data)):
    print(f"\n{d_name}:")
    for fw, val in zip(frameworks, d_data):
        print(f"  {fw:<18} {val}")

In [ ]:
# 选型决策树
print("=== 选型决策树 ===")
print()
print("你的需求是什么?")
print()
print("1. 需要RAG (知识库问答)?")
print("   是 -> LlamaIndex (首选) 或 LangChain")
print()
print("2. 需要多Agent协作?")
print("   是 -> AutoGen (对话式) 或 CrewAI (角色式)")
print()
print("3. 需要完整的Agent能力 (工具+链+记忆)?")
print("   是 -> LangChain (最全面)")
print()
print("4. 快速原型, 不想写代码?")
print("   是 -> Dify (低代码平台)")
print()
print("5. 企业级, 需要.NET/Java集成?")
print("   是 -> Semantic Kernel")
print()
print("6. RFFI场景推荐:")
print("   简单认证API:    FastAPI + 直接模型调用")
print("   知识库问答:     LlamaIndex + RFFI知识库")
print("   完整Agent系统:  LangChain + RFFI工具集")
print("   多Agent安全审计: AutoGen/CrewAI + RFFI专家团队")

In [ ]:
# 框架组合策略
print("=== 框架组合策略 ===")
print()
print("实际项目中, 常组合使用多个框架:")
print()
print("方案1: LangChain + LlamaIndex")
print("  LangChain: Agent编排 + 工具调用")
print("  LlamaIndex: 知识检索 (作为LangChain的检索工具)")
print("  适合: 需要RAG的Agent系统")
print()
print("方案2: LangChain + FastAPI")
print("  LangChain: Agent逻辑")
print("  FastAPI: 部署为API服务")
print("  适合: 生产部署")
print()
print("方案3: AutoGen + LangChain Tools")
print("  AutoGen: 多Agent协作")
print("  LangChain: 提供工具定义")
print("  适合: 多Agent + 丰富工具")
print()
print("方案4: 纯自研 (最灵活)")
print("  自己写Agent循环 + 工具调用 + 记忆")
print("  适合: 特殊需求 / 学习目的 / 极致控制")

---
## 6. 从零构建轻量Agent框架

理解框架原理最好的方式是自己写一个。

In [ ]:
# 轻量Agent框架
class Tool:
    def __init__(self, name: str, description: str, func: Callable, param_schema: Dict = None):
        self.name = name
        self.description = description
        self.func = func
        self.param_schema = param_schema or {}
    
    def run(self, **kwargs) -> str:
        try:
            return str(self.func(**kwargs))
        except Exception as e:
            return f"Error: {e}"

class Memory:
    def __init__(self, max_messages: int = 20):
        self.messages: List[Dict] = []
        self.max_messages = max_messages
    
    def add(self, role: str, content: str):
        self.messages.append({"role": role, "content": content})
        if len(self.messages) > self.max_messages:
            self.messages = self.messages[-self.max_messages:]
    
    def get_context(self, last_n: int = 10) -> str:
        return "\n".join([f"[{m['role']}] {m['content']}" for m in self.messages[-last_n:]])

class LiteAgent:
    def __init__(self, name: str = "Agent", system_prompt: str = ""):
        self.name = name
        self.system_prompt = system_prompt
        self.tools: Dict[str, Tool] = {}
        self.memory = Memory()
        self.max_steps = 10
    
    def add_tool(self, tool: Tool):
        self.tools[tool.name] = tool
    
    def _get_tools_description(self) -> str:
        return "\n".join([f"- {name}: {t.description}" for name, t in self.tools.items()])
    
    def _think(self, observation: str) -> Dict:
        return self._rule_based_think(observation)
    
    def _rule_based_think(self, observation: str) -> Dict:
        obs_lower = observation.lower()
        
        if not self.tools:
            return {"thought": "没有可用工具, 直接回答", "action": "finish", "answer": observation}
        
        for tool_name, tool in self.tools.items():
            keywords = tool_name.replace("_", " ").split()
            if any(kw in obs_lower for kw in keywords):
                return {"thought": f"使用{tool_name}工具", "action": "tool", "tool": tool_name, "args": {}}
        
        first_tool = list(self.tools.keys())[0]
        return {"thought": f"尝试使用{first_tool}", "action": "tool", "tool": first_tool, "args": {}}
    
    def run(self, task: str) -> str:
        self.memory.add("user", task)
        observation = task
        
        for step in range(self.max_steps):
            decision = self._think(observation)
            self.memory.add("thought", decision["thought"])
            
            if decision["action"] == "finish":
                answer = decision.get("answer", observation)
                self.memory.add("assistant", answer)
                return answer
            
            if decision["action"] == "tool":
                tool_name = decision["tool"]
                if tool_name in self.tools:
                    result = self.tools[tool_name].run(**decision.get("args", {}))
                    self.memory.add("tool", f"{tool_name} -> {result}")
                    observation = result
                    
                    if step >= 2:
                        self.memory.add("assistant", f"最终结果: {result}")
                        return result
                else:
                    observation = f"Tool {tool_name} not found"
        
        return "Max steps reached"

print("LiteAgent 框架创建完成")

In [ ]:
# 使用LiteAgent构建RFFI Agent
def scan_rf(frequency: float = 2400.0) -> str:
    n = np.random.randint(2, 6)
    return json.dumps({"devices": n, "freq_mhz": frequency})

def fingerprint_rf(device_id: int = 0) -> str:
    conf = np.random.uniform(0.7, 0.99)
    return json.dumps({"device_id": device_id, "confidence": round(conf, 4), "auth": conf > 0.7})

def detect_jamming(power_dbm: float = -40.0) -> str:
    return json.dumps({"power": power_dbm, "jammed": power_dbm > -50})

agent = LiteAgent(
    name="RFFI Agent",
    system_prompt="你是射频安全专家, 负责设备发现、认证和安全检测"
)

agent.add_tool(Tool("scan_rf", "扫描射频信号发现设备", scan_rf, {"frequency": {"type": "float"}}))
agent.add_tool(Tool("fingerprint_rf", "提取射频指纹认证设备", fingerprint_rf, {"device_id": {"type": "int"}}))
agent.add_tool(Tool("detect_jamming", "检测射频干扰", detect_jamming, {"power_dbm": {"type": "float"}}))

# 运行
tasks = ["scan rf signal", "fingerprint rf device", "detect jamming"]
for task in tasks:
    result = agent.run(task)
    print(f"任务: {task}")
    print(f"结果: {result}")
    print()

In [ ]:
# 框架演进趋势
print("=== Agent 框架演进趋势 ===")
print()
print("1. 从硬编码到LLM驱动")
print("   早期: if-else规则")
print("   现在: LLM自主决策")
print("   未来: 模型自主学习使用工具")
print()
print("2. 从单Agent到多Agent")
print("   单Agent: 简单任务")
print("   多Agent: 复杂系统, 分工协作")
print("   未来: 自组织Agent网络")
print()
print("3. 从提示词工程到自动优化")
print("   手动调prompt")
print("   DSPy: 自动优化prompt")
print("   未来: Agent自动进化")
print()
print("4. 从云端到边缘")
print("   大模型在云端")
print("   小模型在边缘 (SLM)")
print("   RFFI场景: 边缘部署 + 云端协作")
print()
print("5. 从通用到垂直")
print("   通用Agent (如ChatGPT)")
print("   垂直Agent (如RFFI安全Agent)")
print("   未来: 领域专用Agent框架")

---
## 总结

### 框架速查

| 框架 | 一句话 | 安装 | 文档 |
|------|--------|------|------|
| LangChain | 最全的Agent框架 | `pip install langchain` | python.langchain.com |
| LlamaIndex | 最强RAG框架 | `pip install llama-index` | docs.llamaindex.ai |
| AutoGen | 微软多Agent | `pip install pyautogen` | microsoft.github.io/autogen |
| CrewAI | 角色扮演Agent | `pip install crewai` | docs.crewai.com |
| Semantic Kernel | 微软企业级 | `pip install semantic-kernel` | learn.microsoft.com |
| Dify | 低代码平台 | Docker部署 | docs.dify.ai |

### RFFI场景推荐

```
简单认证API:     FastAPI + 模型直接调用
知识库问答:      LlamaIndex + RFFI知识库
完整Agent:       LangChain + RFFI工具集
多Agent安全审计:  CrewAI/AutoGen + RFFI专家团队
生产部署:        LangChain + FastAPI + Docker
```

### 核心设计原则

1. **工具优先**: Agent的能力来自工具, 先设计好工具
2. **提示词是关键**: 好的system prompt决定Agent行为
3. **记忆很重要**: 没有记忆的Agent是健忘的
4. **安全第一**: 限制工具权限, 防止Agent失控
5. **简单开始**: 先用最简单的方案, 再逐步复杂化